# 第 7 课：Batch、Padding、Length 与 Mask

真实语音长短不同，但神经网络通常希望一个 batch 是规则矩阵。本课学习如何补齐长度，以及如何告诉模型哪些位置是真实数据、哪些只是 padding。

路线：不同长度 WAV → waveform batch → lengths → mask → Log-Mel batch → masked statistics → padding 浪费与 bucketing。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 张量与编码器 |
| 建议投入 | 2～4 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 6 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | padding、length、mask |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：padding、length、mask。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import ipywidgets as widgets
from IPython.display import Audio,display

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PARTS=ROOT/'data'/'spoken_digits_parts'

## 1. 为什么 batch 会遇到长度问题？

一个 batch 中有四条语音：`0`、`1`、`6`、`8`。每条都是 8 kHz，但说话时长不同，因此采样点数量不同。NumPy 不能直接把不同长度的一维数组堆成普通二维矩阵。

In [ ]:
digits=[0,1,6,8]
audios=[]
sample_rate=None
for digit in digits:
    x,sr=sf.read(PARTS/f'{digit}_jackson_0.wav',dtype='float32')
    if x.ndim>1:x=x.mean(axis=1)
    audios.append(x); sample_rate=sr
    print(f'digit {digit}: {len(x)} samples = {len(x)/sr:.3f} s')
display(Audio(audios[0],rate=sample_rate))

In [ ]:
fig,axes=plt.subplots(len(audios),1,figsize=(12,8),sharex=True)
for ax,digit,x in zip(axes,digits,audios):
    t=np.arange(len(x))/sample_rate
    ax.plot(t,x,linewidth=0.7); ax.set_ylabel(str(digit)); ax.grid(alpha=0.2)
axes[-1].set_xlabel('Time (s)'); axes[0].set_title('四条真实语音：长度不同')
plt.tight_layout(); plt.show()

### 热身题

1. 哪个数字的音频最长？哪个最短？
2. 这些音频能否直接 `np.stack(audios)`？
3. 它们采样率相同，为什么数组长度仍然不同？

## 2. Padding：补到本 batch 的最大长度

设最长音频长度为 $L_{max}$，较短音频末尾补 0。结果 shape 为 `(batch, time_samples)`。补零只是为了组成矩阵，不代表多出了一段真实静音。

In [ ]:
def pad_waveforms(audios,pad_value=0.0):
    lengths=np.array([len(x) for x in audios],dtype=np.int64)
    max_length=int(lengths.max())
    batch=np.full((len(audios),max_length),pad_value,dtype=np.float32)
    for i,x in enumerate(audios):
        batch[i,:len(x)]=x
    return batch,lengths

wave_batch,wave_lengths=pad_waveforms(audios)
print('batch shape:',wave_batch.shape)
print('lengths:',wave_lengths)
print('padding samples:',wave_batch.shape[1]-wave_lengths)

In [ ]:
plt.figure(figsize=(12,4))
plt.imshow(wave_batch,aspect='auto',cmap='coolwarm',vmin=-0.7,vmax=0.7)
for i,L in enumerate(wave_lengths):plt.axvline(L-0.5,color='yellow',linewidth=1)
plt.xlabel('Sample index'); plt.ylabel('Batch item')
plt.yticks(range(len(digits)),digits); plt.title('Waveform batch：黄线右侧是 padding')
plt.colorbar(label='Amplitude'); plt.show()

### 练习 1

1. batch 的第二维为什么等于最长音频？
2. 对数字 8，黄线右侧的 0 是真实录音吗？
3. 如果 padding 到整个数据集最长音频，而不是当前 batch 最长音频，会发生什么？

## 3. Length：每条数据真正有多长

`lengths[i]` 保存第 i 条音频的真实采样点数。仅有 padded batch 不足以恢复真实边界，因为真实语音中也可能恰好出现数值 0。

In [ ]:
for digit,L in zip(digits,wave_lengths):
    print(f'digit {digit}: valid samples [0, {L}), padded samples [{L}, {wave_batch.shape[1]})')

## 4. Mask：把长度展开成真假矩阵

常见约定：`True/1` 表示真实位置，`False/0` 表示 padding。

$$mask[b,t]=(t<length[b])$$

In [ ]:
def lengths_to_mask(lengths,max_length=None):
    max_length=int(lengths.max()) if max_length is None else max_length
    positions=np.arange(max_length)[None,:]
    return positions<lengths[:,None]

wave_mask=lengths_to_mask(wave_lengths,wave_batch.shape[1])
print('mask shape:',wave_mask.shape)
print('valid counts:',wave_mask.sum(axis=1))
assert np.array_equal(wave_mask.sum(axis=1),wave_lengths)

In [ ]:
plt.figure(figsize=(12,3.5))
plt.imshow(wave_mask,aspect='auto',cmap='gray_r',interpolation='nearest')
plt.xlabel('Sample index'); plt.ylabel('Batch item')
plt.yticks(range(len(digits)),digits); plt.title('Waveform mask：亮色是真实采样，暗色是 padding')
plt.show()

### 练习 2

1. `wave_mask.shape` 与 `wave_batch.shape` 是否相同？
2. 为什么 `mask.sum(axis=1)` 应当等于 lengths？
3. 有些库用 1 表示 padding，而不是有效位置。使用 API 前应该做什么？

## 5. 不使用 mask 会造成什么错误？

以平均绝对幅值为例：如果直接对 padded batch 求平均，较短音频会混入更多补零，因此平均值被人为拉低。

In [ ]:
naive_mean=np.mean(np.abs(wave_batch),axis=1)
masked_mean=(np.abs(wave_batch)*wave_mask).sum(axis=1)/wave_mask.sum(axis=1)
fig,ax=plt.subplots(figsize=(9,4))
xpos=np.arange(len(digits)); width=0.35
ax.bar(xpos-width/2,naive_mean,width,label='错误：包含 padding')
ax.bar(xpos+width/2,masked_mean,width,label='正确：只统计 valid')
ax.set_xticks(xpos,digits); ax.set_xlabel('Digit'); ax.set_ylabel('Mean absolute amplitude')
ax.set_title('Padding 会污染统计量'); ax.legend(); ax.grid(axis='y',alpha=0.2)
plt.show()

正确的 masked mean：

$$mean_b=\frac{\sum_t x_{b,t}mask_{b,t}}{\sum_t mask_{b,t}}$$

## 6. 从不同长度 WAV 提取不同长度 Log-Mel

即使 Mel 维度固定为 40，时间帧数仍随音频时长变化。单条特征 shape 是 `(time_frames, mel_bins)`。

In [ ]:
def extract_logmel(audio,sr=8000):
    mel=librosa.feature.melspectrogram(y=audio,sr=sr,n_fft=256,win_length=200,hop_length=80,
        window='hann',center=False,power=2,n_mels=40)
    return librosa.power_to_db(mel,ref=np.max,top_db=80).T.astype(np.float32)

features=[extract_logmel(x,sample_rate) for x in audios]
for digit,feat in zip(digits,features):print(f'digit {digit}: feature shape {feat.shape}')

In [ ]:
fig,axes=plt.subplots(1,len(features),figsize=(15,4),sharey=True)
for ax,digit,feat in zip(axes,digits,features):
    ax.imshow(feat.T,origin='lower',aspect='auto',vmin=-80,vmax=0,cmap='magma')
    ax.set_title(f'{digit}: T={len(feat)}'); ax.set_xlabel('Time frame')
axes[0].set_ylabel('Mel bin'); plt.tight_layout(); plt.show()

## 7. Padding Log-Mel 特征

Log-Mel 使用相对 dB 时，0 dB 是最强值，所以不能随意用 0 表示‘安静’。本例用 -80 dB padding，并且仍必须提供 mask。很多模型会先做均值方差归一化，此后也可能用 0 padding；真正可靠的是 mask，而不是猜 padding 数值。

In [ ]:
def pad_features(features,pad_value=-80.0):
    lengths=np.array([len(x) for x in features],dtype=np.int64)
    max_time=int(lengths.max()); feature_dim=features[0].shape[1]
    batch=np.full((len(features),max_time,feature_dim),pad_value,dtype=np.float32)
    for i,x in enumerate(features):batch[i,:len(x)]=x
    mask=lengths_to_mask(lengths,max_time)
    return batch,lengths,mask

feature_batch,feature_lengths,feature_mask=pad_features(features)
print('feature batch (B,T,F):',feature_batch.shape)
print('feature lengths:',feature_lengths)
print('feature mask:',feature_mask.shape)

In [ ]:
fig,axes=plt.subplots(len(digits),1,figsize=(12,9),sharex=True,sharey=True)
for i,(ax,digit) in enumerate(zip(axes,digits)):
    ax.imshow(feature_batch[i].T,origin='lower',aspect='auto',vmin=-80,vmax=0,cmap='magma')
    ax.axvline(feature_lengths[i]-0.5,color='cyan',linewidth=1.5)
    ax.set_ylabel(f'{digit}')
axes[-1].set_xlabel('Time frame'); axes[0].set_title('Padded Log-Mel：青线右侧是 -80 dB padding')
plt.tight_layout(); plt.show()

### 关键 shape

- waveform batch：`(B, samples)`
- waveform lengths：`(B,)`
- waveform mask：`(B, samples)`
- feature batch：`(B, T, F)`
- feature lengths：`(B,)`
- feature mask：`(B, T)`

这里 B 是 batch size，T 是时间帧数，F 是 Mel bins。

## 8. Waveform length 与 feature length 不相同

一条音频有 L 个采样点，经过分帧后只有 T 个特征帧。使用 `center=False`、Librosa 的 FFT frame 长度为 n_fft 时：

$$T=1+\left\lfloor\frac{L-n\_fft}{hop}\right\rfloor$$

模型的 encoder 通常使用 feature lengths 或 feature mask，而不是原始 waveform lengths。

In [ ]:
predicted_feature_lengths=1+(wave_lengths-256)//80
print('公式预测:',predicted_feature_lengths)
print('实际结果:',feature_lengths)
print('完全一致:',np.array_equal(predicted_feature_lengths,feature_lengths))

## 9. Attention mask 与 padding mask 命名陷阱

不同框架约定可能相反：

- `attention_mask` 常见约定：1/True 是有效位置。
- `key_padding_mask` 在某些 API 中：True 反而表示需要忽略的 padding。

因此不能只看变量名，必须阅读函数文档，并用一个小例子验证。

In [ ]:
attention_mask=feature_mask                 # True = valid
key_padding_mask=~feature_mask             # True = ignore/padding
print('valid mask 第一条:',attention_mask[0,:10])
print('padding mask 第一条:',key_padding_mask[0,:10])
assert not np.any(attention_mask & key_padding_mask)

## 10. Padding 浪费与 Bucketing

若把很短和很长的语音放在同一 batch，短语音需要补很多 padding。训练时常按长度分桶（bucketing），把相近长度样本放一起。

In [ ]:
def show_padding_waste(selected_count=4):
    chosen=sorted(wave_lengths[:selected_count])
    total_slots=len(chosen)*max(chosen)
    valid=sum(chosen); padding=total_slots-valid
    print(f'valid samples: {valid}')
    print(f'padding samples: {padding}')
    print(f'padding 比例: {padding/total_slots:.1%}')
    plt.figure(figsize=(9,3))
    for i,L in enumerate(chosen):
        plt.barh(i,L,label='valid' if i==0 else None)
        plt.barh(i,max(chosen)-L,left=L,color='lightgray',label='padding' if i==0 else None)
    plt.xlabel('Samples'); plt.ylabel('Batch item'); plt.title('Batch padding waste'); plt.legend(); plt.show()
widgets.interact(show_padding_waste,selected_count=widgets.IntSlider(value=4,min=2,max=4,step=1,description='Batch size'))

## 11. 一个可复用的 collate 函数

训练数据加载器通常调用 collate function，把多条变长特征整理成 batch。

In [ ]:
def collate_logmel(features,pad_value=-80.0):
    batch,lengths,mask=pad_features(features,pad_value)
    return {'features':batch,'lengths':lengths,'attention_mask':mask}

model_inputs=collate_logmel(features)
for name,value in model_inputs.items():print(name,value.shape,value.dtype)
assert model_inputs['features'].shape[:2]==model_inputs['attention_mask'].shape
assert np.array_equal(model_inputs['attention_mask'].sum(1),model_inputs['lengths'])

## 本课测试

1. 为什么不同长度音频不能直接 stack？
2. padding 的目的是什么？
3. padding 出来的 0 是否属于真实录音？
4. lengths 保存什么？
5. mask 为什么不能简单地通过 `x != 0` 推断？
6. 常见 valid mask 中 True 表示什么？
7. 不使用 mask 求平均会造成什么偏差？
8. Log-Mel 相对 dB 特征中，为什么 0 不适合表示安静 padding？
9. feature batch `(B,T,F)` 三个维度分别是什么？
10. waveform length 和 feature length 为什么不同？
11. attention mask 与 key padding mask 的 True 语义是否总相同？
12. bucketing 为什么能节省计算？
13. 如果模型经过时间下采样，mask/length 是否还需要更新？
14. 写出一个 batch 应至少携带的三个字段。

参考答案：1 shape 不一致；2 组成规则矩阵；3 不属于；4 每条样本真实长度；5 真实音频也会有 0；6 有效位置；7 短样本被更多 padding 拉低；8 0 dB 是最强值；9 batch/时间/特征；10 分帧把多个采样点变成一帧；11 不一定；12 相近长度放一起减少补齐；13 需要；14 features、lengths、attention_mask。

## 小结

`不同长度 features → padding → batch (B,T,F) + lengths (B,) + mask (B,T)`

padding 负责形状，mask 负责语义。模型必须知道哪些位置是真的。

下一课：进入 PyTorch，用 Tensor 表示 batch，并实现一个最小的声学编码器，观察输入 `(B,T,F)` 如何变成隐藏表示 `(B,T,D)`。

<!-- course-upgrade-v2 -->
## 强化练习：第 7 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `padding`、`length`、`mask`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**把补零区域错误地计入均值**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现 lengths_to_mask 并测试边界**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**说明 length 如何贯穿 CTC loss**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：padding、length、mask。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 padding、length、mask。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
